# 01 · Read the raw data

**Needs:** the four CSV files in `data/raw/`. **Produces:** nothing, this notebook is only for reading and checking.

The patents were extracted from Google BigQuery with four SQL queries, one per WIPO sector.
Here we read the files, look at their structure, and run some basic checks.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

RAW = Path("../data/raw")
files = sorted(RAW.glob("patents_*.csv"))
for f in files:
    print(f.name)

patents_chemistry_2021-2026_n2000.csv
patents_electrical_engineering_2021-2026_n2000.csv
patents_instruments_2021-2026_n2000.csv
patents_mechanical_engineering_2021-2026_n2000.csv


## 1. What a CSV file looks like

A CSV is a text file. We print the first 600 characters of one file.

In [2]:
with open(files[0], encoding="utf-8") as f:
    print(f.read(600))

wipo_field_number,wipo_field_name,wipo_sector,publication_number,publication_date,title_en,abstract_en,ipc_codes
14,Organic fine chemistry,Chemistry,AU-2016364014-B2,20210401,"Preparation of latanoprostene bunod of desired, pre-defined quality by gravity chromatography","The subject of the invention is a process for the preparation of Latanoprostene bunod of formula (I) with a purity higher than 95% where chromatography is used applying normal phase gravity silicagel column chromatography where the used silicagel is irregular silicagel or spherical silicagel an as eluent and eluent mixture con


The first line is the header. Every other record starts with the field number.
Text with commas is wrapped in double quotes, and the abstract can span **several lines**.
This is why the number of lines in the file is larger than the number of patents:

In [3]:
n_lines = sum(1 for _ in open(files[0], encoding="utf-8"))
df_one = pd.read_csv(files[0], dtype=str)
print("lines in the file:", n_lines)
print("rows read by pandas:", len(df_one))

lines in the file: 24476
rows read by pandas: 22000


`pandas` understands the quotes, so it returns one row per patent.

We read all columns as text (`dtype=str`): we convert dates and numbers ourselves in notebook 02.

## 2. Read one file

In [4]:
df_one.head(3)

,wipo_field_number,wipo_field_name,wipo_sector,publication_number,publication_date,title_en,abstract_en,ipc_codes
0,14,Organic fine chemistry,Chemistry,AU-2016364014-B2,20210401,"Preparation of latanoprostene bunod of desired, pre-defined quality by gravi...",The subject of the invention is a process for the preparation of Latanoprost...,C07C405/00
1,14,Organic fine chemistry,Chemistry,AU-2017269495-B2,20220310,Method for preparing a composition comprising omega-3 fatty acid salts and a...,The present invention relates to a method for preparing a composition compri...,A23D7/005;A23L33/17;A61K31/202;C07C51/00;C07C57/03;C11C1/02
2,14,Organic fine chemistry,Chemistry,AU-2017291826-B2,20211021,Phosphorus prodrugs of sGC stimulators,The present patent application discloses the compounds according to Formula ...,A61K31/422;A61K31/506;C07F9/09


In [5]:
df_one.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22000 entries, 0 to 21999
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   wipo_field_number   22000 non-null  object
 1   wipo_field_name     22000 non-null  object
 2   wipo_sector         22000 non-null  object
 3   publication_number  22000 non-null  object
 4   publication_date    22000 non-null  object
 5   title_en            22000 non-null  object
 6   abstract_en         22000 non-null  object
 7   ipc_codes           22000 non-null  object
dtypes: object(8)
memory usage: 1.3+ MB


## 3. Read the four files

We keep the name of the file in a new column, so we can always trace a row back to its source.

In [6]:
parts = []
for f in files:
    d = pd.read_csv(f, dtype=str)
    d["source_file"] = f.name
    print(f"{f.name:<62} {d.shape[0]:>6} rows  {d.shape[1]} columns")
    parts.append(d)

# all files must have the same columns before we stack them
assert all(list(p.columns) == list(parts[0].columns) for p in parts)

raw = pd.concat(parts, ignore_index=True)
print("\ntotal:", raw.shape)

patents_chemistry_2021-2026_n2000.csv                           22000 rows  9 columns


patents_electrical_engineering_2021-2026_n2000.csv              16000 rows  9 columns
patents_instruments_2021-2026_n2000.csv                         10000 rows  9 columns


patents_mechanical_engineering_2021-2026_n2000.csv              16000 rows  9 columns

total: (64000, 9)


## 4. Basic checks

**Rows per field.** The queries ask for 2,000 publications per field.

In [7]:
per_field = (
    raw.groupby(["wipo_sector", "wipo_field_number", "wipo_field_name"])
    .size()
    .rename("rows")
    .reset_index()
)
per_field["wipo_field_number"] = per_field["wipo_field_number"].astype(int)
per_field.sort_values("wipo_field_number").reset_index(drop=True)

,wipo_sector,wipo_field_number,wipo_field_name,rows
0,Electrical engineering,1,"Electrical machinery, apparatus, energy",2000
1,Electrical engineering,2,Audio-visual technology,2000
2,Electrical engineering,3,Telecommunications,2000
3,Electrical engineering,4,Digital communication,2000
4,Electrical engineering,5,Basic communication processes,2000
5,Electrical engineering,6,Computer technology,2000
6,Electrical engineering,7,IT methods for management,2000
7,Electrical engineering,8,Semiconductors,2000
8,Instruments,9,Optics,2000
9,Instruments,10,Measurement,2000


Every field has exactly 2,000 rows: all fields had at least 2,000 eligible publications.
The sectors have different sizes because they have a different number of fields.

In [8]:
raw.groupby("wipo_sector").size().rename("rows")

wipo_sector
Chemistry                 22000
Electrical engineering    16000
Instruments               10000
Mechanical engineering    16000
Name: rows, dtype: int64

**Missing values.** In text files, "missing" can be an empty string.

In [9]:
empty = (raw.drop(columns="source_file").isna() | (raw.drop(columns="source_file").apply(lambda s: s.str.strip()) == "")).sum()
empty

wipo_field_number     0
wipo_field_name       0
wipo_sector           0
publication_number    0
publication_date      0
title_en              0
abstract_en           0
ipc_codes             0
dtype: int64

**Duplicates.** The same publication can be selected for two fields.
A (publication, field) pair must be unique, but a publication number can repeat.

In [10]:
print("duplicated (publication, field) pairs:", raw.duplicated(["publication_number", "wipo_field_number"]).sum())
print("distinct publications:", raw["publication_number"].nunique(), "of", len(raw), "rows")

duplicated (publication, field) pairs: 0
distinct publications: 63968 of 64000 rows


In [11]:
twice = raw[raw.duplicated("publication_number", keep=False)].sort_values("publication_number")
twice[["publication_number", "wipo_field_number", "wipo_field_name", "title_en"]].head(10)

,publication_number,wipo_field_number,wipo_field_name,title_en
2193,CN-110305820-B,15,Biotechnology,Lactobacillus rhamnosus CCFM1064 and application thereof
8129,CN-110305820-B,18,Food chemistry,Lactobacillus rhamnosus CCFM1064 and application thereof
10129,CN-111349264-B,19,Basic materials chemistry,Hydrophobic oleophobic antistatic PVC composite film
6126,CN-111349264-B,17,"Macromolecular chemistry, polymers",Hydrophobic oleophobic antistatic PVC composite film
16231,CN-112646191-A,22,Micro-structural and nano-technology,Rare earth-organic framework nano fluorescent probe and preparation method a...
6202,CN-112646191-A,17,"Macromolecular chemistry, polymers",Rare earth-organic framework nano fluorescent probe and preparation method a...
56148,CN-112673133-A,29,Other special machines,Molded body of cellulose fiber and method for producing same
54139,CN-112673133-A,28,Textile and paper machines,Molded body of cellulose fiber and method for producing same
14171,CN-113341486-A,21,"Surface technology, coating",Structurally colored articles and methods of making and using structurally c...
10282,CN-113341486-A,19,Basic materials chemistry,Structurally colored articles and methods of making and using structurally c...


**Dates.** The date is stored as `YYYYMMDD`. We check that it can be converted and look at the range.

In [12]:
dates = pd.to_datetime(raw["publication_date"], format="%Y%m%d", errors="coerce")
print("dates that cannot be parsed:", dates.isna().sum())
print("first:", dates.min().date(), " last:", dates.max().date())

dates that cannot be parsed: 0
first: 2021-01-01  last: 2026-09-17


The queries cover publications up to 30 September 2026, but the latest date in the data is a few days earlier.

**IPC codes.** They are stored in one text column, separated by `;`.

In [13]:
raw["ipc_codes"].head(5).tolist()

['C07C405/00',
 'A23D7/005;A23L33/17;A61K31/202;C07C51/00;C07C57/03;C11C1/02',
 'A61K31/422;A61K31/506;C07F9/09',
 'A23K20/26;A23K50/75;C07H21/04;C07K14/415;C12N9/16;C12N9/24',
 'A61K31/52;A61P35/00;C07D473/16']

In [14]:
n_codes = raw["ipc_codes"].str.count(";") + 1
n_codes.describe()

count    64000.000000
mean         4.192344
std          3.206762
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
max        109.000000
Name: ipc_codes, dtype: float64

## Try it yourself

1. How many patents have more than 10 IPC codes? Which one has the most?
2. Which publication number has the longest title?
3. How many rows come from each patent office (the first part of the publication number)?

## Next

`02_merge_and_clean` turns these four files into one cleaned dataset.